In [20]:
from langgraph.graph import StateGraph,START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated 
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages #builtin function  for reducer basically to add messages 
from langgraph.checkpoint.memory import MemorySaver #save STATE to memory and load it back when needed

In [21]:
load_dotenv()  # Load environment variables from .env file


True

In [22]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages] #base message is a class that represents a message in the conversation. It can be either a human message or an AI message. The messages are stored in a list, which is annotated with a description of what it represents.

In [23]:
model = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")

In [24]:
def chat_node(State: ChatState):
    messages =State['messages']
    response = model.invoke(messages)
    return {'messages': [response]}  

In [25]:
checkpointer = MemorySaver() #create a checkpoint object to save the state to memory and load it back when needed. The argument is the path to the file where the state will be saved.
graph = StateGraph(ChatState)

graph.add_node("chat_node",chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

chatbot = graph.compile(checkpointer=checkpointer) #compile the graph and pass the checkpointer object to it. The checkpointer object will be used to save the state to memory and load it back when needed.


In [26]:
thread_id = '1'
while True:
    user_input = input("Type Here ")
    print("User:", user_input)
    if user_input.strip().lower() in["exit","quit","bye"]:
        break
    config = {'configurable': {'thread_id': thread_id}}  # Create a config dictionary with the thread_id


    # Create a HumanMessage object with the user's input
    
    response = chatbot.invoke({'messages': [HumanMessage(content=user_input)]}, config=config) #whenever invoke is called, state is stated from scratch no previous info is given to it. So, we need to pass the previous messages to it.

    print("Bot:", response['messages'][-1].content)

User: hi my name is nishant
Bot: Hi Nishant, nice to meet you! How can I help you today?
User: what is my name
Bot: Your name is **Nishant**.
User: ok
Bot: Great! So, Nishant, what can I do for you now? Do you have a question, need help with something, or just want to chat?
User: exit


In [28]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='hi my name is nishant', additional_kwargs={}, response_metadata={}, id='fc050179-aa34-4827-a8b6-5439409e8eff'), AIMessage(content='Hi Nishant, nice to meet you! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fbcff-a3d4-7ec0-90f2-fc042abb38b8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 219, 'total_tokens': 227, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 203}}), HumanMessage(content='what is my name', additional_kwargs={}, response_metadata={}, id='24e87640-f3ad-4991-bd71-4c9e7b2f1908'), AIMessage(content='Your name is **Nishant**.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_r